# 6. Salvar e recuperar dados preparados

O pré-processamento de EEG é caro. Filtragem, resampling e windowding em um sujeito pode levar segundos. Fazer isso toda vez que se reinicia o programa vai fazer o projeto levar muito mais tempo do que deveria.

Assim, podemos salvar o processamento feito e reutilizá-lo.

## 6.1 Requisitos

In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

import braindecode
import eegdash
from braindecode.datasets import BaseConcatDataset, RawDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing import create_fixed_length_windows
from eegdash.viz import use_eegdash_style

use_eegdash_style()
SEED = 42
np.random.seed(SEED)
mne.set_log_level("ERROR")
print(
    f"eegdash {eegdash.__version__} | braindecode {braindecode.__version__} | "
    f"numpy {np.__version__}"
)

/home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


eegdash 0.9.1 | braindecode 1.8.1 | numpy 2.5.3


/home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/home/arielalves/Desktop/Iniciacao-cientifica-EEG-FM/EEGDash-Tutorials/Translated-Tutorials/.venv/lib/python3.12/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


## 6.2 Arquivos de cache vivos fora do código: um modelo mental

Primeiro, a árvore de artefatos que esse tutorial produz:

```text
<cache_root>/
  windows/                 # FIF: one .fif per child + JSON sidecars
    0/0-raw.fif            #   raw samples + info
    0/description.json     #   subject / task / run
    0/window_kwargs.json   #   how the windows were cut
  windows.zarr/            # optional: Zarr-chunked, blosc-compressed
  features.parquet         # tabular: one row per window, typed cols
```

Segundo, o caminho de leitura. `load_concat_datasaet(windows_path, preload=True)` reidrata a árvore FIF em uma classe `braindecode.datasets.BaseConcatDataset` com os metadados do frame intactos.

## 6.3 Construindo um pequeno dataset com janelas

Vamos simular o EEG de um sujeito (2 canais, 4s a 100Hz) e dividir isso em janelas não sobrepostas de 2s.

In [2]:
SFREQ, WIN_S = 100, 2
# Simulated signal
signal = np.random.randn(2, 4 * SFREQ).astype("float32") * 1e-6
# Simulated info
info = mne.create_info(["Cz", "Pz"], sfreq=SFREQ, ch_types="eeg")
# Complete recording (signal + info)
recording = RawDataset(
    mne.io.RawArray(signal, info),
    description={"subject": "S01", "task": "rest"},
)
# Create windows
windows = create_fixed_length_windows(
    BaseConcatDataset([recording]),
    start_offset_samples=0,
    stop_offset_samples=None,
    window_size_samples=WIN_S * SFREQ,
    window_stride_samples=WIN_S * SFREQ,
    drop_last_window=True,
    preload=True,
)

n_windows = len(windows)
sample_shape = windows[0][0].shape
n_channels, window_samples = int(sample_shape[0]), int(sample_shape[1])

pd.Series(
    {
        "n_windows": n_windows,
        "windows[0][0].shape": str(tuple(sample_shape)),
        "X.dtype": str(np.asarray(windows[0][0]).dtype),
        "child class": type(windows.datasets[0]).__name__,
    },
    name="value",
).to_frame()

/tmp/ipykernel_157409/1671822262.py:12: DeprecationWarning: `drop_last_window` is deprecated and will be removed in version 2.0. Use `on_last_window='drop'` if True or `on_last_window='overlap'` if False. See https://github.com/braindecode/braindecode/pull/1058 for feedback.
  windows = create_fixed_length_windows(


,value
n_windows,2
windows[0][0].shape,"(2, 200)"
X.dtype,float32
child class,EEGWindowsDataset


## 6.4 Salvar as janelas em FIF

`braindecode.datasets.BaseConcatDataset.save` escreve um subdiretório para cada dataset filho, cada um com um `raw.fif` (ou `epo.fif`) mais um JSON de apoio para descrição, nome do target e argumentos de processamento.

In [ ]:
# Obtiain /tmp/eegdash_save_
cache_root = Path(tempfile.mkdtemp(prefix="eegdash_save_"))
windows_path = cache_root / "windows"

# Start time counting to write
t0 = time.perf_counter()
# Save windows!
windows.save(str(windows_path), overwrite=True)
# Ends time counting
fif_write_s = time.perf_counter() - t0

# Counts files size
def _dir_size_bytes(path: Path) -> int:
    """Sum file sizes under ``path`` recursively (Windows-portable, no ``du``)."""
    total = 0
    for root, _, files in os.walk(path):
        for name in files:
            total += (Path(root) / name).stat().st_size
    return total


fif_size_mb = _dir_size_bytes(windows_path) / 1e6
saved_files = sorted(
    p.relative_to(cache_root).as_posix() for p in windows_path.rglob("*")
)

print(f"saved: {windows_path}")
print(f"artifact tree (first 6): {saved_files[:6]}")
print(f"FIF write_s={fif_write_s:.4f} s, size_mb={fif_size_mb:.4f}")

saved: /tmp/eegdash_save_6ahlxzsf/windows
artifact tree (first 6): ['windows/0', 'windows/0/0-raw.fif', 'windows/0/description.json', 'windows/0/metadata_df.pkl', 'windows/0/raw_preproc_kwargs.json', 'windows/0/window_kwargs.json']
FIF write_s=0.0051 s, size_mb=0.0054


## 6.5 Recuperar as janelas

Utiliza-se `load_concat_dataset(windows_path, preload=True)` Exatamente como abaixo. `preload=True` retorna um array `float32` na RAM.

In [9]:
# Start recovering time counting
t0 = time.perf_counter()
# Loads windows
reloaded_fif = load_concat_dataset(windows_path, preload=True)
# Ends time counting
fif_read_s = time.perf_counter() - t0

print(
    f"reload OK: type={type(reloaded_fif).__name__}, n={len(reloaded_fif)}, "
    f"read_s={fif_read_s:.4f}"
)

reload OK: type=BaseConcatDataset, n=2, read_s=0.0068


Vamos investigar se os dados são os mesmos.

In [10]:
x_orig = np.asarray(windows[0][0]).copy()
x_re = np.asarray(reloaded_fif[0][0]).copy()
residual = x_re - x_orig

assert len(reloaded_fif) == n_windows, "reloaded window count differs"
assert x_re.shape == sample_shape, "reloaded window shape differs"
assert np.allclose(x_re, x_orig, atol=1e-7), "samples drifted beyond float32 tol"
assert list(reloaded_fif.description.columns) == list(windows.description.columns)

print(
    f"shapes match: original={sample_shape}, reloaded={x_re.shape}; "
    f"max|residual|={float(np.max(np.abs(residual))):.2e}"
)

shapes match: original=(2, 200), reloaded=(2, 200); max|residual|=0.00e+00


## 6.6 Cache Zarr opcional (acesso randomizado em blocos)

FIF é bom para uma gravação, mas o custo de acesso cresce linearment com o tamanho. o Zarr armazena blocos de tamanho fio e lê qualquer janela em dezenas de milissegundos para centenas de GB. 

In [ ]:
# Verify if feature existss
try:
    BaseConcatDataset._convert_to_zarr_inline  # noqa: B018 - feature probe
    has_zarr = True
except (AttributeError, ImportError):
    has_zarr = False

# Try to store with Zarr cache
zarr_record = None
if has_zarr:
    zarr_path = cache_root / "windows.zarr"
    try:
        # Applies Zarr cache in the windows and Stores it.
        t0 = time.perf_counter()
        windows._convert_to_zarr_inline(
            zarr_path,
            compression="blosc",
            compression_level=5,
            chunk_size=5_000_000
        )
        zarr_write_s = time.perf_counter() - t0

        # Reloads windows with Zarr cache
        t0 = time.perf_counter()
        reloaded_zarr = type(windows)._load_from_zarr_inline(zarr_path, preload=True)
        zarr_read_s = time.perf_counter() - t0
        zarr_size_mb = _dir_size_bytes(zarr_path) / 1e6
        zarr_record = {
            "name": "windows.zarr (Zarr)",
            "write_s": zarr_write_s,
            "read_s": zarr_read_s,
            "size_mb": zarr_size_mb,
        }
        print(
            f"Zarr write_s={zarr_write_s:.4f}, read_s={zarr_read_s:.4f}, "
            f"size_mb={zarr_size_mb:.4f}"
        )
    except (ImportError, RuntimeError) as exc:
        has_zarr = False
        print(f"Zarr extra unavailable, skipping: {type(exc).__name__}: {exc}")
else:
    print("Zarr extra not installed (pip install braindecode[hub]); skipping.")

## 6.7 Salvar e recarregar uma tabela de features tabulares

Ao invés de consumir os sinais puros, muitos notebooks de downstream consomem `(n_windows, n_features)`. O parquet atende essa necessidade: Comprimido e legível em R, Julia, Python e DuckDB. Calculamos uma feature por canal por janela (média por canal) e garantimos que os tipos de dados sejam preservados, já que é uma propriedade da qual o armazenamento de features depende.

In [ ]:
features = pd.DataFrame(
    [
        {
            "Cz_mean": float(windows[i][0][0].mean()),
            "Pz_mean": float(windows[i][0][1].mean()),
            "window_idx": i,
        }
        for i in range(n_windows)
    ]
)
features_path = cache_root / "features.parquet"
t0 = time.perf_counter()
features.to_parquet(features_path, index=False)
parquet_write_s = time.perf_counter() - t0

t0 = time.perf_counter()
features_back = pd.read_parquet(features_path)
parquet_read_s = time.perf_counter() - t0

pd.testing.assert_frame_equal(features_back, features)
parquet_size_mb = features_path.stat().st_size / 1e6
print(f"feature table dtype:\n{features.dtypes.to_string()}")
print(
    f"Parquet write_s={parquet_write_s:.4f}, read_s={parquet_read_s:.4f}, "
    f"size_mb={parquet_size_mb:.4f}"
)
features.head()

## 6.8 Montar o livro-razão de formatos de gravação

Toda linha da tabela abaixo alimenta o painel 1 da figura final. Os valores são vivos, nada é hard codado. `has_zarr=False` omite a linha do Zarr.

In [ ]:
format_records = [
    {
        "name": "windows/ (FIF)",
        "write_s": fif_write_s,
        "read_s": fif_read_s,
        "size_mb": fif_size_mb,
    },
]
if zarr_record is not None:
    format_records.append(zarr_record)
format_records.append(
    {
        "name": "features.parquet",
        "write_s": parquet_write_s,
        "read_s": parquet_read_s,
        "size_mb": parquet_size_mb,
    }
)
records_df = pd.DataFrame(format_records)
records_df["write_ms"] = (records_df["write_s"] * 1000).round(2)
records_df["read_ms"] = (records_df["read_s"] * 1000).round(2)
records_df[["name", "write_ms", "read_ms", "size_mb"]]

## 6.9 Provenience stamp

O cache sobrevive ao código. Registramos o que um leitor futuro precisa para executar novamente o pipeline upstream: versões de pacotes, seed e git shor-SHA. A chamada `subprocess.run` funciona corretamente quando o git está indisponível.

In [19]:
def _git_short_sha() -> str:
    """Return the current git short-SHA, or a fallback string when git is missing."""
    try:
        result = subprocess.run(
            ["git", "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            timeout=2.0,
            check=False,
        )
        sha = (result.stdout or "").strip()
        return sha or "git: not available"
    except (OSError, subprocess.SubprocessError):
        return "git: not available"


provenance = {
    "eegdash": eegdash.__version__,
    "braindecode": braindecode.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "mne": mne.__version__,
    "seed": str(SEED),
    "git": _git_short_sha(),
}
pd.Series(provenance, name="value").to_frame()

,value
eegdash,0.9.1
braindecode,1.8.1
numpy,2.5.3
pandas,3.0.5
mne,1.13.2
seed,42
git,525cc7e


## 6.10 Um erro comum: Esquecer overwrite=True

Esquecer `overwrite=True` é comum ao rodar pela segunda vez este tutorial.
Um segundo erro é chamar `load_concat_dataset` em um caminho que está sem os sidecars (JSON de apoio).

In [ ]:
try:
    windows.save(str(windows_path), overwrite=False)
except (FileExistsError, OSError, RuntimeError) as exc:
    print(f"Caught {type(exc).__name__}: {str(exc)[:80]}")
    shutil.rmtree(windows_path)
    windows.save(str(windows_path), overwrite=False)
    print("Recovery: rmtree + save without overwrite=True succeeded.")

# Wrong directory layout (no sidecars). load_concat_dataset rejects it.
broken = cache_root / "broken_layout"
broken.mkdir(parents=True, exist_ok=True)
try:
    load_concat_dataset(broken, preload=True)
except (FileNotFoundError, IndexError, KeyError, ValueError) as exc:
    print(
        f"Recovery: load_concat_dataset rejected broken layout "
        f"({type(exc).__name__}: {str(exc)[:60]})."
    )